# Instella GPU Suite — one cell per model

**How to use:**
1. Run **Cell 1 (Setup)** once. Re-run it only if the Colab runtime restarts.
2. Run **one model cell at a time** (Cells 2–5). Each is an independent ~2–3 h run that
   saves to Google Drive as it goes.
3. If a model cell disconnects, just **re-run that same cell** — it resumes item-by-item
   from Drive (skips everything already done; at most ~8 items are ever repeated).
4. Run **Cell 6 (Status)** anytime to see progress across all models.

Finished models stay finished on Drive even if you never run the other cells. Order doesn't
matter, but Instruct → Math first gives the most useful results soonest.

Prereqs: Runtime → GPU · Secrets (🔑): `github` + `HF_TOKEN` (Notebook-access ON).

## Cell 1 — Setup (run once per session)

In [ ]:
LIMIT = 200   # items per benchmark. Same value in every cell below.

import os, subprocess
from google.colab import drive, userdata
import torch
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > GPU, then restart."
print("GPU:", torch.cuda.get_device_name(0))

drive.mount('/content/drive')
RUN_DIR = f"/content/drive/MyDrive/instella-gpu-suite-L{LIMIT}"
os.environ["GITHUB_TOKEN"] = userdata.get("github")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["HF_HUB_DISABLE_XET"] = "1"

if not os.path.isdir("/content/Instella-Reasoning"):
    !git clone -b agent/cpu-experiment-suite https://$GITHUB_TOKEN@github.com/GIND123/Instella-Reasoning /content/Instella-Reasoning
%cd /content/Instella-Reasoning
!git pull
if subprocess.run(["which", "instella-reasoning"], capture_output=True).returncode != 0:
    !pip install -q -e ".[hf,retrieval,viz,stats]"
    !pip install -q "bitsandbytes>=0.46.1"
print("\nSetup done. Results dir on Drive:", RUN_DIR)

## Cell 2 — Instella-3B-Instruct  (run this first; re-run if it disconnects)

In [ ]:
%cd /content/Instella-Reasoning
!SUITE_MODELS="instruct" bash experiments/run_gpu_suite.sh {LIMIT} "{RUN_DIR}"

## Cell 3 — Instella-3B-Math  (downloads ~6 GB the first time)

In [ ]:
%cd /content/Instella-Reasoning
!SUITE_MODELS="math" bash experiments/run_gpu_suite.sh {LIMIT} "{RUN_DIR}"

## Cell 4 — Instella-3B (base)

In [ ]:
%cd /content/Instella-Reasoning
!SUITE_MODELS="base" bash experiments/run_gpu_suite.sh {LIMIT} "{RUN_DIR}"

## Cell 5 — AMD-OLMo-1B (scale baseline)

In [ ]:
%cd /content/Instella-Reasoning
!SUITE_MODELS="olmo" bash experiments/run_gpu_suite.sh {LIMIT} "{RUN_DIR}"

## Cell 6 — Status / accuracy (run anytime to see progress)

In [ ]:
import glob, json
BENCH = ["gsm8k", "math", "logiqa2", "arc_challenge", "bbh", "reclor"]
print(f"progress in {RUN_DIR}  (target {LIMIT} items each)\n")
score_files = sorted(f for f in glob.glob(f"{RUN_DIR}/scores/*__*.jsonl") if not f.endswith("__ALL.jsonl"))
if not score_files:
    print("  (nothing scored yet — run a model cell)")
for f in score_files:
    rows = [json.loads(l) for l in open(f) if l.strip()]
    if rows:
        acc = sum(r["correct"] for r in rows) / len(rows)
        done = "OK " if len(rows) >= LIMIT else "..."
        print(f"  {done} {os.path.basename(f):45} {len(rows):4}/{LIMIT}  acc={acc:.3f}")

## Cell 7 — Download a zip (optional; results already persist on Drive)

In [ ]:
!cd /content/drive/MyDrive && zip -qr /content/gpu_results.zip instella-gpu-suite-L{LIMIT} && echo zipped
from google.colab import files
files.download("/content/gpu_results.zip")